In [0]:
spark

In [0]:
jdbc_url = "jdbc:sqlserver://sachcloudy.database.windows.net:1433;database=batch25_test"
table_name = "dbo.EmployeeData"
connection_properties = {
    "user": "admin123",
    "password": "Batch@25",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

df = spark.read.jdbc(url=jdbc_url, table=table_name, properties=connection_properties)
display(df)

In [0]:
query = "SELECT * FROM dbo.EmployeeData"
df_sql = spark.read.jdbc(url=jdbc_url, table=f"({query}) as emp", properties=connection_properties)
display(df_sql)

In [0]:
target_df = df.select('FirstName','LastName','Department','Designation')
target_df.display()

In [0]:
# JDBC write is not supported on serverless compute — using pymssql as workaround
import pymssql

pdf = target_df.toPandas()

conn = pymssql.connect(
    server='sachcloudy.database.windows.net',
    user='admin123',
    password='Batch@25',
    database='batch25_test'
)
cursor = conn.cursor()

# Create table if it doesn't exist
cursor.execute("""
IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'onrole_employee' AND schema_id = SCHEMA_ID('dbo'))
CREATE TABLE dbo.onrole_employee (
    FirstName NVARCHAR(255),
    LastName NVARCHAR(255),
    Department NVARCHAR(255),
    Designation NVARCHAR(255)
)
""")

# Insert rows
for _, row in pdf.iterrows():
    cursor.execute(
        "INSERT INTO dbo.onrole_employee (FirstName, LastName, Department, Designation) VALUES (%s, %s, %s, %s)",
        (row['FirstName'], row['LastName'], row['Department'], row['Designation'])
    )

conn.commit()
conn.close()
print(f"Successfully written {len(pdf)} rows to dbo.onrole_employee")

In [0]:

%pip install pymssql

In [0]:
dbutils.library.restartPython()